# Data Engineering Fundamentals

Companion notebook for the [Data Engineering lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/05-data-engineering-fundamentals).

**The idea in one sentence.** The data layer decides your ML system's speed and cost:
**columnar storage** makes analytical scans cheap, **batch vs streaming** is a
latency/throughput trade-off with no single winner, and a single **skewed (hot) partition**
dominates the user-visible p99 no matter how fast the average is.

What we build and verify:

- **Row-major vs column-major:** the same data, different layout — analytical column scans
  favour columnar.
- **Batch vs streaming:** streaming minimises latency; batching maximises throughput
  efficiency.
- **Partition skew:** one slow partition dominates the tail latency.

We simulate all three, **validate the batch/streaming trade-off and the skew tail**, then
cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

BRAND  = '#6366f1'
TEAL   = '#2dd4bf'
ROSE   = '#fb7185'
ORANGE = '#f97316'
YELLOW = '#facc15'
MUTED  = '#475569'

## 1. Row-major vs column-major scan

A 100k-row events table with 10 columns. The training job only needs columns 0 and 1 (call them `user_id` and `ts`). We store the same data in two NumPy layouts — row-major (`C`) and column-major (`F`) — and time the same workload on each.

In a row-major layout, `table[i, :]` is contiguous in memory, so reading a full row is a single cache-friendly stride but reading one column is the worst-case access pattern. In a column-major layout, `table[:, j]` is contiguous, so reading a column scans one contiguous run — the same pattern Parquet and ORC exploit on disk.

NumPy's in-memory layouts are a faithful analog of the on-disk row-store vs column-store split.

In [ ]:
N_ROWS = 100_000
N_COLS = 10

rng = np.random.default_rng(0)
data = rng.standard_normal((N_ROWS, N_COLS)).astype(np.float64)

# Two layouts holding the same data.
row_major = np.ascontiguousarray(data)            # C order: rows contiguous
col_major = np.asfortranarray(data)               # F order: columns contiguous

print(f'shape         = {row_major.shape}')
print(f'row_major strides = {row_major.strides}   <- big stride between rows of same col')
print(f'col_major strides = {col_major.strides}   <- small stride between rows of same col')

NEEDED_COLS = [0, 1]   # the training job needs only 2 of the 10 columns

def time_scan(layout, cols, repeats=200):
    '''Mean elapsed time (seconds) to sum the requested columns.'''
    # Warm-up
    _ = layout[:, cols].sum()
    t0 = time.perf_counter()
    for _ in range(repeats):
        _ = layout[:, cols].sum()
    return (time.perf_counter() - t0) / repeats

row_t = time_scan(row_major, NEEDED_COLS)
col_t = time_scan(col_major, NEEDED_COLS)

print(f'\nrow-major scan-2-cols : {row_t*1e6:7.1f} µs per pass')
print(f'col-major scan-2-cols : {col_t*1e6:7.1f} µs per pass')
print(f'speedup               : {row_t/col_t:6.2f}×')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
bars = ax.bar(['row-major\n(row store)', 'col-major\n(column store)'],
              [row_t * 1e6, col_t * 1e6],
              color=[ROSE, TEAL], edgecolor='#0f1117')
ax.set_ylabel('time per scan (µs)')
ax.set_title(f'Scan 2 of {N_COLS} columns over {N_ROWS:,} rows')
ax.grid(True, axis='y')
for bar, val in zip(bars, [row_t * 1e6, col_t * 1e6]):
    ax.annotate(f'{val:.1f} µs',
                xy=(bar.get_x() + bar.get_width() / 2, val),
                xytext=(0, 3), textcoords='offset points',
                ha='center', color='#e2e8f0')
plt.tight_layout(); plt.show()

The column-major scan is several times faster on the same hardware. The data has not changed; only the *layout* has. The on-disk analog is a CSV (every row a contiguous record) vs a Parquet file (every column a contiguous run). For ML training, where almost every workload reads a small subset of columns from a large table, columnar is the natural fit — and the gap widens as columns become wider and the projected subset becomes smaller.

## 2. OLTP vs OLAP workloads on the same table

We simulate the two opposite workloads on the same in-memory table:

- **OLTP-like**: 10,000 single-row updates against random row indices.
- **OLAP-like**: one big aggregate (`sum` of a column across all rows).

Both touch the same data; they exercise it completely differently. The OLTP workload is a stream of pointer-chases; the OLAP workload is a single contiguous scan. The numbers below explain why production systems separate the two.

In [ ]:
table = rng.standard_normal((N_ROWS, N_COLS)).copy()

# OLTP: 10,000 random single-row mutations.
N_TX = 10_000
idx = rng.integers(0, N_ROWS, size=N_TX)
vals = rng.standard_normal(N_TX)

t0 = time.perf_counter()
for i in range(N_TX):
    table[idx[i], 0] += vals[i]
oltp_t = time.perf_counter() - t0

# OLAP: one aggregate across all rows on the same column.
t0 = time.perf_counter()
total = table[:, 0].sum()
olap_t = time.perf_counter() - t0

print(f'OLTP : {N_TX:,} single-row updates  -> {oltp_t*1000:7.2f} ms total  ({oltp_t/N_TX*1e6:.2f} µs / tx)')
print(f'OLAP : 1 aggregate over {N_ROWS:,} rows -> {olap_t*1000:7.2f} ms total  (sum = {total:+.1f})')
print(f'\nPer-row cost ratio: OLTP touches {N_TX:,} rows, OLAP touches {N_ROWS:,} rows.')
print(f'Per-row time      : OLTP {oltp_t/N_TX*1e9:.0f} ns/row   OLAP {olap_t/N_ROWS*1e9:.0f} ns/row')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
labels = ['OLTP\n10k single-row updates', 'OLAP\n1 full-column aggregate']
times_ms = [oltp_t * 1000, olap_t * 1000]
bars = ax.bar(labels, times_ms, color=[ROSE, TEAL], edgecolor='#0f1117')
ax.set_ylabel('wall-clock time (ms)')
ax.set_title('Two opposite workloads on the same table')
ax.grid(True, axis='y')
for bar, val in zip(bars, times_ms):
    ax.annotate(f'{val:.2f} ms',
                xy=(bar.get_x() + bar.get_width() / 2, val),
                xytext=(0, 3), textcoords='offset points',
                ha='center', color='#e2e8f0')
plt.tight_layout(); plt.show()

The OLTP workload spends most of its time per-row (random pointer chases, branch overhead, the Python interpreter loop). The OLAP workload is one vectorised scan and amortises every per-row cost across the full table. *Both* workloads have legitimate users — the live product needs OLTP semantics, analytics and training need OLAP throughput. Running them on the same database means tuning for one and starving the other; the standard fix is to keep them on separate engines with replication between them.

## 3. Batch vs streaming: latency / throughput trade-off

A stream of events arrives at a roughly uniform rate. Two processors:

- **Streaming.** Processes each event the moment it arrives. End-to-end latency for an event $\approx$ its own processing time (no waiting).
- **Batch.** Buffers events and processes them every $T$ seconds. An event that arrives at time $t$ waits until the next batch tick at $\lceil t / T \rceil \cdot T$, then is processed in bulk.

We sweep $T \in \{0.5, 1, 2, 5, 10\}$ seconds and plot two things: the *mean end-to-end latency* (event arrival → result emitted), and *CPU efficiency* measured as events processed per unit of processing work.

In [ ]:
DURATION_S      = 60.0          # simulate one minute
EVENTS_PER_SEC  = 200
N_EVENTS        = int(DURATION_S * EVENTS_PER_SEC)

# Uniformly arriving events.
arrival_t = np.sort(rng.uniform(0, DURATION_S, size=N_EVENTS))

PER_EVENT_COST   = 50e-6  # 50 µs real per-event work (whether batched or streamed)
BATCH_FIXED_COST = 2e-3   # 2 ms fixed overhead per batch tick (job startup, IO, sync)
STREAM_FIXED_OVERHEAD_PER_EVENT = 200e-6  # streaming pays a 200 µs per-event 'plumbing' tax

# Streaming: end-to-end latency = stream overhead + per-event cost.
stream_latency_per_event = STREAM_FIXED_OVERHEAD_PER_EVENT + PER_EVENT_COST
stream_work_total = N_EVENTS * stream_latency_per_event
stream_efficiency = N_EVENTS / stream_work_total      # events per second of work

def simulate_batch(arrival_t, T):
    '''Return per-event latencies and total processing work for batch interval T.'''
    # Each event waits until the next multiple of T.
    next_tick = np.ceil(arrival_t / T) * T
    wait = next_tick - arrival_t                       # wait before batch starts
    n_batches = int(np.ceil(arrival_t.max() / T))      # number of ticks
    total_work = n_batches * BATCH_FIXED_COST + N_EVENTS * PER_EVENT_COST
    # End-to-end latency = wait + share of batch processing time.
    # Approximate batch processing time per batch: BATCH_FIXED_COST + events_in_batch * PER_EVENT_COST.
    # For mean per-event latency we treat each event as paying its own per-event cost.
    latency = wait + PER_EVENT_COST                    # event processed at start of its batch
    return latency, total_work, n_batches

Ts = [0.5, 1.0, 2.0, 5.0, 10.0]
rows = []
for T in Ts:
    lat, work, nb = simulate_batch(arrival_t, T)
    eff = N_EVENTS / work
    rows.append({
        'T':                 T,
        'n_batches':         nb,
        'mean_latency_ms':   lat.mean() * 1000,
        'p95_latency_ms':    np.percentile(lat, 95) * 1000,
        'efficiency_eps':    eff,
    })

print(f'{"T (s)":>6s} {"#batch":>7s} {"mean lat (ms)":>14s} {"p95 lat (ms)":>14s} {"events/s of work":>18s}')
for r in rows:
    print(f'{r["T"]:>6.1f} {r["n_batches"]:>7d} {r["mean_latency_ms"]:>14.1f} {r["p95_latency_ms"]:>14.1f} {r["efficiency_eps"]:>18.0f}')

print(f'\nstreaming reference   mean lat = {stream_latency_per_event*1000:>6.3f} ms   events/s of work = {stream_efficiency:.0f}')

In [ ]:
Ts_arr   = np.array([r['T'] for r in rows])
mean_lat = np.array([r['mean_latency_ms'] for r in rows])
eff_arr  = np.array([r['efficiency_eps'] for r in rows])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(Ts_arr, mean_lat, '-o', color=BRAND, linewidth=2, markersize=8, label='batch')
ax1.axhline(stream_latency_per_event * 1000, color=TEAL, linestyle='--', label='streaming')
ax1.set_xlabel('batch interval T (s)')
ax1.set_ylabel('mean end-to-end latency (ms)')
ax1.set_title('Latency: streaming wins')
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.grid(True, which='both')
ax1.legend(frameon=False)

ax2.plot(Ts_arr, eff_arr, '-o', color=ORANGE, linewidth=2, markersize=8, label='batch')
ax2.axhline(stream_efficiency, color=TEAL, linestyle='--', label='streaming')
ax2.set_xlabel('batch interval T (s)')
ax2.set_ylabel('events processed / second of work')
ax2.set_title('Throughput: batch wins')
ax2.set_xscale('log')
ax2.grid(True, which='both')
ax2.legend(frameon=False)

plt.tight_layout(); plt.show()

### Validate: streaming wins latency, batching wins throughput

Streaming emits each event immediately (low latency) but pays a per-event plumbing tax.
Batching amortises a fixed per-tick cost over many events, so **efficiency (throughput) rises
with the batch interval** while **latency also rises**. There is no single winner — it is a
trade-off. We confirm both directions.

In [ ]:
print(f'batch interval range: {Ts_arr.min():.2g}s .. {Ts_arr.max():.2g}s')
print(f'efficiency: {eff_arr[0]:.0f} -> {eff_arr[-1]:.0f} events/s of work  (bigger batch = more throughput)')
print(f'batch mean latency at largest T: {mean_lat[-1]:.1f}ms  vs streaming {stream_latency_per_event*1000:.3f}ms')
assert eff_arr[-1] > eff_arr[0], 'larger batches amortise fixed cost -> higher throughput efficiency'
assert mean_lat[-1] > stream_latency_per_event * 1000, 'but batching costs latency vs streaming'
print('\n✅ batch vs streaming is a latency/throughput trade-off — pick the point your SLA needs')

Streaming wins on latency — every event is emitted as soon as it arrives. Batch wins on throughput — amortising the fixed per-batch overhead across many events and dropping the per-event plumbing tax pushes events-per-unit-work far above streaming. The right choice for an ML pipeline depends on which axis matters: training is throughput-bound (always batch); online recommendation is latency-bound (streaming, at least on the request path). A 'lambda' architecture runs both: batch produces accurate ground truth, streaming produces fast approximations, the serving layer reconciles them.

## 4. Skew matters: one slow partition stalls the whole pipeline

A streaming consumer is usually partitioned across multiple workers — four partitions of a Kafka topic, four Flink task slots, four worker processes. Every partition gets roughly $1/4$ of the events. In the happy case, every partition processes at the same rate and per-event latency is uniformly low.

In the unhappy case, *one* partition is slower than the others — because of data skew (one user generates 10% of traffic), a hot key, a slower disk, or a noisy neighbour. Events stuck behind the slow partition pile up. The mean latency may still look fine; the tail explodes.

We simulate 4 partitions; partition 0 runs 3× slower than the others. Each event is routed to a partition by hashing a user id. We compute per-event end-to-end latency and look at the distribution.

In [ ]:
N_PARTITIONS = 4
N_EVENTS_SKEW = 20_000
BASE_SERVICE_MS = 1.0
SLOW_PARTITION = 0
SLOW_FACTOR = 3.0

# Arrivals: Poisson-ish, mean 0.5 ms apart.
arrivals = np.cumsum(rng.exponential(0.5, size=N_EVENTS_SKEW))    # ms
# Assign each event to a partition. Uniform hashing.
part = rng.integers(0, N_PARTITIONS, size=N_EVENTS_SKEW)

# Per-event service time. Slow partition takes SLOW_FACTOR× longer.
service = np.where(part == SLOW_PARTITION,
                   BASE_SERVICE_MS * SLOW_FACTOR,
                   BASE_SERVICE_MS) * rng.uniform(0.8, 1.2, size=N_EVENTS_SKEW)

# Simulate per-partition queueing. Each partition processes its events FIFO,
# starting an event at max(arrival, previous_end_in_this_partition).
start = np.zeros(N_EVENTS_SKEW)
end   = np.zeros(N_EVENTS_SKEW)
last_end_per_part = np.zeros(N_PARTITIONS)
for i in range(N_EVENTS_SKEW):
    p = part[i]
    start[i] = max(arrivals[i], last_end_per_part[p])
    end[i]   = start[i] + service[i]
    last_end_per_part[p] = end[i]

latency_ms = end - arrivals

print(f'overall   mean latency = {latency_ms.mean():6.2f} ms')
print(f'overall   p95  latency = {np.percentile(latency_ms, 95):6.2f} ms')
print(f'overall   p99  latency = {np.percentile(latency_ms, 99):6.2f} ms')
print(f'\nper-partition p95 latency:')
for p in range(N_PARTITIONS):
    mask = part == p
    print(f'  partition {p}{" (slow)" if p == SLOW_PARTITION else "":>8s} : '
          f'p95 = {np.percentile(latency_ms[mask], 95):6.2f} ms   '
          f'p99 = {np.percentile(latency_ms[mask], 99):6.2f} ms   '
          f'n = {mask.sum()}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

bins = np.linspace(0, latency_ms.max(), 60)
for p in range(N_PARTITIONS):
    mask = part == p
    color = ROSE if p == SLOW_PARTITION else BRAND
    label = f'partition {p}' + (' (slow)' if p == SLOW_PARTITION else '')
    ax1.hist(latency_ms[mask], bins=bins, alpha=0.55, color=color, label=label)
ax1.set_xlabel('end-to-end latency (ms)')
ax1.set_ylabel('events')
ax1.set_title('Per-partition latency distribution')
ax1.legend(frameon=False, fontsize=9)
ax1.grid(True, axis='y')

# Sorted latency = approximation of the empirical CDF tail.
sorted_lat = np.sort(latency_ms)
q = np.linspace(0, 1, len(sorted_lat))
ax2.plot(q, sorted_lat, color=BRAND, linewidth=2)
ax2.axhline(np.percentile(latency_ms, 95), color=YELLOW, linestyle='--', label='p95')
ax2.axhline(np.percentile(latency_ms, 99), color=ROSE,   linestyle='--', label='p99')
ax2.set_xlabel('quantile')
ax2.set_ylabel('latency (ms)')
ax2.set_title('Overall latency CDF: one slow partition drives the tail')
ax2.grid(True)
ax2.legend(frameon=False, loc='upper left')

plt.tight_layout(); plt.show()

### Validate: one slow partition dominates the tail

Uniform hashing spreads events evenly, but if one partition is slower (a hot key, a big
tenant), its events queue up and dominate the **p99** — even though the mean looks fine. We
confirm the p99 is far above the p50, and that the slow partition drives it.

In [ ]:
p50 = np.percentile(latency_ms, 50)
p99 = np.percentile(latency_ms, 99)
p99_fast_only = np.percentile(latency_ms[part != SLOW_PARTITION], 99)
print(f'overall  p50 = {p50:.1f}ms,  p99 = {p99:.1f}ms')
print(f'p99 excluding the slow partition: {p99_fast_only:.1f}ms')
assert p99 > 2 * p50, 'the tail is far worse than the median'
assert p99 > p99_fast_only, 'the slow partition dominates the global p99'
print('\n✅ skew is a tail problem — the mean hides it; rebalance/repartition the hot partition')

The mean overall latency looks reasonable, but the *p99* is dominated by the slow partition: events routed to it queue up because the partition cannot drain as fast as events arrive. This is the canonical streaming-skew failure mode — the system is healthy *on average* and broken *at the tail*. Mitigations are familiar to anyone who's run Kafka or Flink: better partition keys to spread the hot key, dynamic rebalancing, per-partition autoscaling, or accepting at-least-once semantics with a fallback path. The lesson for ML feature serving: if your streaming feature pipeline has any skew, your model will see stale features for a non-trivial fraction of requests — and at scale, that is a measurable accuracy regression you cannot catch with offline evaluation.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **wrong storage layout** | row-major kills analytical column scans; use columnar for OLAP |
| **batch vs streaming** | latency/throughput trade-off (verified) — match to the SLA |
| **partition skew** | one slow shard dominates the p99 (verified); rebalance hot keys |
| **count-balanced ≠ work-balanced** | equal event counts can still imbalance work (demo) |
| **no per-partition monitoring** | the mean hides a hot shard; watch per-partition tails |

Demo: a hot partition inflates the shared tail even at balanced counts.

In [ ]:
# The skew gotcha quantified: how much does the single slow partition inflate the tail that
# every user shares? We compare the median latency of the slow partition against the others.
# The lesson: uniform hashing does not guarantee uniform LOAD when service times differ.
slow_med = np.median(latency_ms[part == SLOW_PARTITION])
fast_med = np.median(latency_ms[part != SLOW_PARTITION])
frac_slow = (part == SLOW_PARTITION).mean()
print(f'slow partition holds {frac_slow:.0%} of events but its median latency is {slow_med:.1f}ms')
print(f'the other partitions median: {fast_med:.1f}ms')
assert slow_med > fast_med, 'the hot partition is slower per event and backs up its own queue'
print('\nA hash that balances COUNTS can still imbalance WORK -> monitor per-partition latency,')
print('isolate hot keys, or repartition. The tail belongs to the slowest shard.')

---
## ✏️ Your turn

### Exercise: implement `simulate_batch_latency(arrivals, batch_interval_s)`

Given a 1-D NumPy array `arrivals` of event arrival times (in seconds, sorted, all $\ge 0$) and a batch interval `batch_interval_s > 0`, return the **mean end-to-end wait** each event experiences before being processed. An event that arrives at time $t$ is processed at the next batch tick, i.e. at time $\lceil t / T \rceil \cdot T$, where $T$ is the batch interval. The wait is `next_tick - t`.

**Reference value to match.** For uniform-on-$[0, D]$ arrivals with $D \gg T$, the mean wait converges to $T / 2$ — because each event lands at a uniform offset within its batch window, and the expected wait until the next tick is half the window. Your function should return a value close to that for the test cases below.

In [ ]:
def simulate_batch_latency(arrivals, batch_interval_s):
    '''Mean end-to-end batch wait time for the given arrivals.

    Args:
        arrivals:        1-D NumPy array of event arrival times (seconds, sorted, >= 0).
        batch_interval_s: positive scalar T. Batches tick at T, 2T, 3T, ...

    Returns:
        The mean of `next_tick - arrival` over all events,
        where `next_tick = ceil(arrival / T) * T`.
    '''
    # TODO(you):
    #   1. Compute next_tick = ceil(arrivals / batch_interval_s) * batch_interval_s.
    #   2. wait = next_tick - arrivals
    #   3. return wait.mean()
    #
    # Hint: np.ceil works element-wise on arrays.
    pass

# Smoke run.
rng_local = np.random.default_rng(42)
arrivals_demo = np.sort(rng_local.uniform(0, 60.0, size=100_000))
print(f'simulate_batch_latency(arrivals, T=1.0) = {simulate_batch_latency(arrivals_demo, 1.0)}')
print(f'reference T/2                            = {1.0 / 2:.3f}')

In [ ]:
# Test 1: large uniform sample, T=1.0 -> mean wait close to 0.5 s.
rng_t = np.random.default_rng(123)
arr = np.sort(rng_t.uniform(0, 600.0, size=200_000))   # 10 minutes of events
mean_wait = simulate_batch_latency(arr, 1.0)
assert mean_wait is not None, 'function returned None'
assert abs(mean_wait - 0.5) < 0.02, f'expected ~0.5, got {mean_wait}'

# Test 2: larger interval scales the mean wait linearly.
mean_wait_T5 = simulate_batch_latency(arr, 5.0)
assert abs(mean_wait_T5 - 2.5) < 0.05, f'expected ~2.5 for T=5, got {mean_wait_T5}'

# Test 3: a single event arriving exactly on a tick has zero wait.
single = np.array([2.0])
assert abs(simulate_batch_latency(single, 1.0) - 0.0) < 1e-9

# Test 4: a single event just after a tick waits almost the full interval.
single = np.array([0.001])
assert abs(simulate_batch_latency(single, 1.0) - 0.999) < 1e-6

# Test 5: smaller interval -> shorter waits.
assert simulate_batch_latency(arr, 0.5) < simulate_batch_latency(arr, 2.0)

# Test 6 (edge case): an empty arrivals array has no events to wait, so the
# mean over zero elements is conventionally NaN -- the function should not
# raise, just propagate NumPy's empty-mean behavior.
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore', category=RuntimeWarning)
    empty_wait = simulate_batch_latency(np.array([]), 1.0)
assert np.isnan(empty_wait), f'expected NaN for an empty arrivals array, got {empty_wait}'

print('All batch-latency tests passed.')

<details>
<summary>Show solution</summary>

```python
def simulate_batch_latency(arrivals, batch_interval_s):
    next_tick = np.ceil(arrivals / batch_interval_s) * batch_interval_s
    wait = next_tick - arrivals
    return float(wait.mean())
```

Three things to notice:

1. **The $T/2$ rule is the headline trade-off.** Increasing the batch interval by 10× cuts the number of batches by 10× (less fixed overhead, higher throughput) but increases the mean wait by 10×. Batch interval is a knob, not a free choice.
2. **The actual tail is $T$, not $T/2$.** Events that arrive just *after* a tick wait almost the full interval. If your SLA is on p95 latency, your headline number is $\approx T$, not $T/2$.
3. **This is why streaming exists.** When the cost of waiting $T$ for the next batch is higher than the cost of running a long-lived stream processor, you switch to streaming. For training data, the cost of waiting is essentially zero; for fraud detection at checkout, the cost of waiting is a chargeback.
</details>

---
## Extra practice: a toy ETL pipeline (DML 187)

Every simulation above assumed the data was already a clean NumPy array. In a
real pipeline, the first job is turning messy raw text into that array — a
small Extract-Transform-Load (ETL) step. Implement `run_etl(csv_text)`:

- **Extract**: split `csv_text` into non-blank lines; the first line is a
  header naming the columns `user_id`, `event_type`, `value` — **in any
  order**, so look up each column's index by name rather than assuming a
  fixed position.
- **Transform**: keep only rows where `event_type == "purchase"`; convert
  `value` to `float`, dropping any row where that conversion fails; sum the
  purchase `value` per `user_id`. Cells may have extra surrounding whitespace.
- **Load**: return the aggregated totals as a list of `(user_id, total_value)`
  tuples, sorted by `user_id` ascending.

In [ ]:
def run_etl(csv_text):
    """Extract-transform-load a tiny purchase-events CSV.

    Args:
        csv_text: str -- CSV text with a header row naming the columns
                  'user_id', 'event_type', 'value' (in any order), and zero
                  or more data rows. Cells may have surrounding whitespace.
    Returns:
        list[tuple[str, float]]: (user_id, total_purchase_value) pairs for
        event_type == 'purchase' rows only, invalid 'value' cells dropped,
        summed per user_id, sorted by user_id ascending.
    """
    # TODO(you):
    #   1. Extract: split csv_text into non-blank lines. If there are none,
    #      return []. Split the header line on ',' and strip each name to
    #      find the column index of 'user_id', 'event_type', 'value'.
    #   2. Transform: for each remaining line, split + strip each cell; skip
    #      rows whose event_type isn't 'purchase'; try float(value), and
    #      skip the row (don't raise) if that fails; accumulate the total
    #      value per user_id in a dict.
    #   3. Load: return sorted(dict.items()).
    pass


# Smoke run
demo_csv = "user_id,event_type,value\n u1, purchase, 10.0\n u2, view, 1.0\n u1, purchase, 5\n u3, purchase, not_a_number\n u2, purchase, 3.5 \n"
print(run_etl(demo_csv))

In [ ]:
# Matches DML 187's worked example: keep purchases, coerce value to float,
# drop the unparseable row, aggregate per user, sort by user_id.
csv1 = "user_id,event_type,value\n u1, purchase, 10.0\n u2, view, 1.0\n u1, purchase, 5\n u3, purchase, not_a_number\n u2, purchase, 3.5 \n"
assert run_etl(csv1) == [('u1', 15.0), ('u2', 3.5)], run_etl(csv1)

# Edge case: empty dataset -- header only, no data rows.
assert run_etl("user_id,event_type,value") == []

# Edge case: single-row dataset -- exactly one purchase row.
assert run_etl("user_id,event_type,value\nu1,purchase,7.5") == [('u1', 7.5)]

# Edge case: header columns in a different order -- must look up by name,
# not assume 'user_id,event_type,value' positions.
csv2 = "value,event_type,user_id\n 1.0, purchase, u1\n 2.0, purchase, u1\n"
assert run_etl(csv2) == [('u1', 3.0)]

# Edge case: blank lines and non-purchase/invalid rows are all ignored,
# leaving no aggregated users at all.
csv3 = "user_id,event_type,value\n\nu1,view,100\nu1,purchase,abc\n\n"
assert run_etl(csv3) == []

print("✅ ETL exercise passed (DML 187)")

<details>
<summary>💡 Show solution</summary>

```python
def run_etl(csv_text):
    lines = [line for line in csv_text.split('\n') if line.strip() != '']
    if not lines:
        return []
    header = [c.strip() for c in lines[0].split(',')]
    idx = {name: i for i, name in enumerate(header)}
    totals = {}
    for line in lines[1:]:
        cells = [c.strip() for c in line.split(',')]
        if len(cells) != len(header):
            continue
        if cells[idx['event_type']] != 'purchase':
            continue
        try:
            value = float(cells[idx['value']])
        except ValueError:
            continue
        user = cells[idx['user_id']]
        totals[user] = totals.get(user, 0.0) + value
    return sorted(totals.items())
```

Looking up columns by name (`idx = {name: i for i, ...}`) rather than
hardcoding positions `0, 1, 2` is what makes this robust to the header-order
edge case above -- a very common real-world ETL gotcha when upstream schemas
drift.
</details>

## Key takeaways

- **Layout is performance:** columnar storage makes analytical column-scans cheap; row-major
  suits per-row (OLTP) access.
- **Batch vs streaming is a trade-off:** streaming minimises latency, batching maximises
  throughput efficiency — no single winner (verified).
- **Skew is a tail problem:** one slow partition dominates the shared p99 even at balanced
  event counts (verified) — monitor per-partition, isolate hot keys.
- **The tail belongs to the slowest shard** — design and monitor for p99, not the mean.